In [1]:
import platform
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchmetrics
from torchvision import datasets, transforms

# Device
if platform.system() == "Darwin":
    device = torch.device("mps" if torch.backends.mps.is_built() else "cpu")
else:
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)

# Config
DATA_DIR    = Path("../data/split")
IMG_SIZE    = 128
BATCH_SIZE  = 32
NUM_WORKERS = 0
NUM_CLASSES = 10
DROPOUT     = 0.5

BASELINE_PATH = "../models/best_run10.pt"   # ungewichtet (run 12)
WEIGHTED_PATH = "../models/best_run13.pt"   # gewichtet (run 13)

mps


In [2]:
eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
])

val_set    = datasets.ImageFolder(DATA_DIR / "val", transform=eval_transform)
val_loader = torch.utils.data.DataLoader(
    val_set, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

print("Val-Bilder:", len(val_set))

Val-Bilder: 381


In [3]:
class CoverCNN(nn.Module):
    def __init__(self, num_classes=10, dropout=0.5):
        super().__init__()
        self.conv1 = nn.Conv2d(3,   32,  kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32,  64,  kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64,  128, kernel_size=3, padding=1)
        self.pool          = nn.MaxPool2d(kernel_size=2, stride=2)
        self.adaptive_pool = nn.AdaptiveAvgPool2d((4, 4))
        self.fc1     = nn.Linear(128 * 4 * 4, 256)
        self.dropout = nn.Dropout(dropout)
        self.fc2     = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = self.adaptive_pool(x)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

In [4]:
def per_class_val(checkpoint_path):
    m = CoverCNN(num_classes=NUM_CLASSES, dropout=DROPOUT).to(device)
    m.load_state_dict(torch.load(checkpoint_path, map_location=device))
    m.eval()
    acc_class = torchmetrics.Accuracy(task="multiclass", num_classes=NUM_CLASSES, average=None).to(device)
    overall   = torchmetrics.Accuracy(task="multiclass", num_classes=NUM_CLASSES).to(device)
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            pred = m(images)
            acc_class.update(pred, labels)
            overall.update(pred, labels)
    return overall.compute().item(), acc_class.compute().cpu().numpy()

names = val_set.classes
ov_base, per_base = per_class_val(BASELINE_PATH)
ov_wt,   per_wt   = per_class_val(WEIGHTED_PATH)

print(f"{'Klasse':18s} {'Baseline':>9} {'Gewichtet':>10} {'Δ':>8}")
print("-" * 47)
for i, n in enumerate(names):
    print(f"{n:18s} {per_base[i]:>9.3f} {per_wt[i]:>10.3f} {per_wt[i]-per_base[i]:>+8.3f}")
print("-" * 47)
print(f"{'GESAMT':18s} {ov_base:>9.3f} {ov_wt:>10.3f} {ov_wt-ov_base:>+8.3f}")

Klasse              Baseline  Gewichtet        Δ
-----------------------------------------------
alternative_rock       0.053      0.184   +0.132
classical              0.129      0.323   +0.194
country                0.121      0.182   +0.061
hiphop                 0.400      0.343   -0.057
house                  0.122      0.244   +0.122
indie_rock             0.051      0.231   +0.179
jazz                   0.341      0.146   -0.195
metal                  0.659      0.463   -0.195
reggae                 0.432      0.405   -0.027
techno                 0.378      0.156   -0.222
-----------------------------------------------
GESAMT                 0.276      0.265   -0.010
